<a href="https://colab.research.google.com/github/Sruthi051006/sruthi-codeboosters-2026/blob/main/Day-4/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#cell - 1 -- installing pyspark , to run without the comments we use pyspark
!pip install pyspark --quiet

print('pyspark installation completed')

pyspark installation completed


In [3]:
#cell - 2 -- import pyspark modules and create sparksession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import  year, month , to_date , col , round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


# Create sparkSession
spark = SparkSession.builder \
    .appName('Day4_BigData_sales') \
    .config('spark.sql.adaptive.enable', 'true') \
    .getOrCreate()
print(f'Spark version' , {spark.version})
print(f'SparkSESSION  :ACTIVE')

print(f'Application : {spark.sparkContext.appName}')



Spark version {'4.0.2'}
SparkSESSION  :ACTIVE
Application : Day4_BigData_sales


In [5]:
# load csv into pyspark dataframe

df_bronze = spark.read \
    .option('header' , 'true') \
    .option('inferSchema' , 'true') \
    .csv('large_sales_data.csv')

print(' ===bronze layer raw data===')

print(f'Rows: {df_bronze.count()}')

print(f'columns: {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')
print()
df_bronze.printSchema()

 ===bronze layer raw data===
Rows: 5000
columns: 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [9]:
#cell 4 -- inspect rows , to use large column names truncate=False
print('first 5 rows')
df_bronze.show(5 , truncate=False)

print('\nbasic statistics for numeric columns')
df_bronze.select('quantity' , 'unit_price' , 'revenue').describe().show()

df_bronze.tail(5)

first 5 rows
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Kum

[Row(order_id=5996, customer_name='Ananya Das', product='Mouse', category='Accessories', quantity=13, unit_price=800, revenue=10400, order_date=datetime.date(2023, 11, 18), city='Bangalore', region='South', sales_rep='Meera Patel', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id=5997, customer_name='Suresh Rao', product='Webcam', category='Accessories', quantity=9, unit_price=2500, revenue=22500, order_date=datetime.date(2023, 6, 7), city='Chennai', region='South', sales_rep='Sunita Rao', payment_method='Credit Card', order_status='Delivered'),
 Row(order_id=5998, customer_name='Arjun Nair', product='Webcam', category='Accessories', quantity=1, unit_price=2500, revenue=2500, order_date=datetime.date(2023, 4, 7), city='Jaipur', region='North', sales_rep='Kavya Reddy', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id=5999, customer_name='Arjun Nair', product='Laptop', category='Electronics', quantity=14, unit_price=45000, revenue=630000, order

In [11]:
#cell 5 -- save bronze layer as parquet , parquet size shrinks  ,,, brone just stores data

df_bronze.write \
   .mode('overwrite') \
   .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: sales_bronze.parquet')

#compare file size
import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) / 1024
  total = 0
  for dirpath  , dirnames , filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath , f))
  return total / 1024

# Change .scv to .csv
csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - parquet_size/csv_size) * 100

print(f'\n CSV size : { csv_size:.1f} KB' )
print(f'parquet size: {parquet_size:.1f} KB')
print(f'reduction: {reduction:.1f}% smaller')
print(f'\nAt 1 TB scale: CSV=1000 GB -> parquet={1000*(1-reduction/100):.0f} GB')


Bronze Parquet saved: sales_bronze.parquet

 CSV size : 529.3 KB
parquet size: 55.1 KB
reduction: 89.6% smaller

At 1 TB scale: CSV=1000 GB -> parquet=104 GB


In [12]:
df_bronze.select("product" , "category" , "quantity").show(5)

+----------+-----------+--------+
|   product|   category|quantity|
+----------+-----------+--------+
|   Monitor|Electronics|      12|
|   Printer|Electronics|      10|
|     Mouse|Accessories|      10|
|    Tablet|Electronics|       5|
|Headphones|Electronics|       4|
+----------+-----------+--------+
only showing top 5 rows


In [13]:
df_bronze.filter(F.col("revenue") > 50000).show()

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|
|    1004|   Suresh Rao| Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West|  Ravi Kumar|Cash on Delivery|  Processing|
|    1008|  Priya Patel| Laptop|Electronics|      13|     45000| 585000|2023-05-29|  Chennai| South| Meera Patel|     Credit Card|   Can

In [14]:
#cell - 6 -- silver - clean and enrich the data

df_silver = df_bronze \
    .dropDuplicates() \
    .dropna(subset=['order_id' , 'product' , 'revenue'])

df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-mm-yyyy')
)

df_silver = df_silver \
     .withColumn('order_year' , year(col('order_date'))) \
     .withColumn('order_month' , month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 40000 , 'high')
     .when(col('revenue') > 10000 , 'medium')
     .otherwise('low')
)
print(f'silver layer rows {df_silver.count()}')
print('new columns added: order_year , order_month , revenue_category')
df_silver.select('product' , 'revenue' , 'order_year' , 'order_month' , 'revenue_category').show(8)

silver layer rows 5000
new columns added: order_year , order_month , revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          medium|
|    Webcam|  17500|      2023|          1|          medium|
|   Speaker|  58500|      2023|          4|            high|
|  Keyboard|   9600|      2023|         12|             low|
|    Laptop| 180000|      2023|          8|            high|
|Headphones|  38500|      2023|          5|          medium|
|    Webcam|  35000|      2023|         11|          medium|
|    Laptop| 360000|      2023|          1|            high|
+----------+-------+----------+-----------+----------------+
only showing top 8 rows


In [15]:
print('Duplicate products and their counts:')
df_bronze.groupBy('product').count().filter(F.col('count') > 1).orderBy(F.col('count').desc()).show(truncate=False)

Duplicate products and their counts:
+----------+-----+
|product   |count|
+----------+-----+
|Webcam    |532  |
|Tablet    |532  |
|USB Hub   |527  |
|Laptop    |502  |
|Keyboard  |495  |
|Mouse     |492  |
|Printer   |488  |
|Monitor   |481  |
|Headphones|481  |
|Speaker   |470  |
+----------+-----+



In [16]:
#cell - 7 save silver layer in parquet

df_silver.write \
   .mode('overwrite') \
   .parquet('sales_silver.parquet')

print('silver Parquet saved: sales_silver.parquet')
print(f'silver size {get_dir_size('sales_silver.parquet'):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')

print(f'read_back rows: {df_verify.count()}(should match the count)')
df_verify.printSchema()

silver Parquet saved: sales_silver.parquet
silver size 59.8 KB
read_back rows: 5000(should match the count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [18]:
# bronze size

df_bronze.write \
   .mode('overwrite') \
   .parquet('sales_bronze.parquet')

print('bronze Parquet saved: sales_bronze.parquet')
print(f'bronze size {get_dir_size('sales_bronze.parquet'):.1f} KB')

df_verify1 = spark.read.parquet('sales_bronze.parquet')

print(f'read_back rows: {df_verify1.count()}(should match the count)')
df_verify1.printSchema()

bronze Parquet saved: sales_bronze.parquet
bronze size 55.1 KB
read_back rows: 5000(should match the count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)

